# M02 — Ingesta y preparación

[← Anterior](../M01-fundamentos-entorno/02-lab-sesion-spark.ipynb) · [Siguiente →](02-lab-ingesta-csv-json.ipynb)

Leemos las fuentes reales de NovaShop. Inferir schema es **exploración**; producir pide schema explícito.

Labs después: ingesta → tipos → limpieza.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

Ejecuta estas dos celdas. Localizan el repo y dejan una `SparkSession` lista.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m02')
print(spark.version, spark.sparkContext.master)


## CSV: todo entra como texto

Sin schema, Spark trata **todas** las columnas como string. Eso es lo que quieres ver ahora.


In [ ]:
orders_txt = spark.read.option("header", True).csv(str(RAW / "orders.csv"))
print("filas", orders_txt.count())
orders_txt.printSchema()
orders_txt.show(3, truncate=False)


## JSON array vs JSONL

`products.json` es **un** documento (array): hace falta `multiLine=True`.  
`events.jsonl` es una línea = un objeto. Sin `multiLine` el array se parte en `_corrupt_record`.


In [ ]:
products = spark.read.option("multiLine", True).json(str(RAW / "products.json"))
events = spark.read.json(str(RAW / "events.jsonl"))
print("products", products.count(), "events", events.count())
products.printSchema()


## Inferencia vs schema

Tipar **no borra** filas. Hay fechas `dd/mm/yyyy`: un solo formato las deja nulas.


In [ ]:
from pyspark.sql.functions import col, coalesce, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType

print("fechas raras (dd/mm/yyyy):")
orders_txt.where(col("OrderDate").contains("/")).select("OrderId", "OrderDate").show()

schema = StructType([
    StructField("OrderId", StringType(), True),
    StructField("CustomerId", StringType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Channel", StringType(), True),
])
orders = (
    spark.read.option("header", True).schema(schema).csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
    .withColumn(
        "order_ts",
        coalesce(
            to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
            to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
        ),
    )
    .drop("order_ts_raw")
)
orders.printSchema()
print("nulos de fecha", orders.where(col("order_ts").isNull()).count())
print("count sigue siendo", orders.count())


**Siguiente:** [lab de ingesta](02-lab-ingesta-csv-json.ipynb) — creas tu notebook y cargas las cuatro fuentes.
